In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [2]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *
from kret_optuna._core.optuna_nb_imports import *
from kret_polars._core.polars_nb_imports import *
from kret_rosetta._core.rosetta_nb_imports import *
from kret_sklearn._core.sklearn_nb_imports import *
from kret_torch_utils._core.torch_nb_imports import *
from kret_tqdm._core.tqdm_nb_imports import *
from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

Loaded environment variables from /Users/Akseldkw/coding/projects_kretsinger/.env
[kret_lgbm._core.lgbm_nb_imports] Imported kret_lgbm._core.lgbm_nb_imports in 2.1090 seconds
[kret_lightning._core.lightning_nb_imports] Imported kret_lightning._core.lightning_nb_imports in 4.0845 seconds
[kret_matplotlib._core.mpl_nb_imports] Imported kret_matplotlib._core.mpl_nb_imports in 0.2776 seconds
[kret_np_pd._core.np_pd_nb_imports] Imported kret_np_pd._core.np_pd_nb_imports in 0.0006 seconds
[kret_optuna._core.optuna_nb_imports] Imported kret_optuna._core.optuna_nb_imports in 0.0022 seconds
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0809 seconds
[kret_rosetta._core.rosetta_nb_imports] Imported kret_rosetta._core.rosetta_nb_imports in 0.0000 seconds
[kret_sklearn._core.sklearn_nb_imports] Imported kret_sklearn._core.sklearn_nb_imports in 0.1802 seconds
[kret_torch_utils._core.torch_nb_imports] Imported kret_torch_utils._core.torch_nb_imports in 0.3666

In [3]:
from heart_nn_loader import HeartNNLoader
from heart_nn import HeartFailureNN

# Load Data

In [4]:
UKS_CONSTANTS.KAGGLEHUB_DIR

PosixPath('/Users/Akseldkw/coding/data_kretsinger/kagglehub')

In [5]:
file_sub_path = "datasets/heart-failure-prediction/versions/1/heart.csv"

In [6]:
UKS_DEFAULTS.READ_CSV_PD_DEFAULT

{'header': 'infer', 'skipinitialspace': True}

In [7]:
df_raw: pd.DataFrame = pd.read_csv(UKS_CONSTANTS.KAGGLEHUB_DIR / file_sub_path, **UKS_DEFAULTS.READ_CSV_PD_DEFAULT)

In [8]:
dtt(df_raw)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
,int64,object,object,int64,int64,int64,object,int64,object,float64,object,int64
289,36,M,ATA,120,166,0,Normal,180,N,0.000,Up,0
371,66,M,ASY,150,0,0,Normal,108,Y,2.000,Flat,1
405,35,M,ASY,120,0,1,Normal,130,Y,1.200,Flat,1
635,67,M,ASY,120,229,0,LVH,129,Y,2.600,Flat,1
716,67,M,ASY,120,237,0,Normal,71,N,1.000,Flat,1


In [9]:
counts = df_raw.value_counts()
counts

Age  Sex  ChestPainType  RestingBP  Cholesterol  FastingBS  RestingECG  MaxHR  ExerciseAngina  Oldpeak  ST_Slope  HeartDisease
28   M    ATA            130        132          0          LVH         185    N               0.0      Up        0               1
58   M    ASY            128        216          0          LVH         131    Y               2.2      Flat      1               1
                         130        0            0          ST          100    Y               1.0      Flat      1               1
                                    263          0          Normal      140    Y               2.0      Flat      1               1
                         132        458          1          Normal      69     N               1.0      Down      0               1
                                                                                                                                 ..
50   M    ASY            150        215          0          Normal      140    Y 

In [10]:
def load_and_clean(base_dir: Path = UKS_CONSTANTS.KAGGLEHUB_DIR, filename: str = file_sub_path):
    df_load = FunctionTransformer(func=pd.read_csv, validate=False, kw_args={})
    custom_cleanup = FunctionTransformer(func=UKS_NP_PD.data_cleanup, validate=False, kw_args={"ret": True})
    pipeline_load_and_clean = PipelinePD(
        steps=[
            ("df_load", df_load),
            ("cleanup_custom", custom_cleanup),
        ]
    )
    df = pipeline_load_and_clean.fit_transform_df(base_dir / filename)
    features, target = UKS_NP_PD.pop_label_and_drop(df, label_col="HeartDisease")
    x_train, x_val, y_train, y_val = train_test_split(features, target, test_size=0.15, random_state=0)
    return x_train, x_val, y_train, y_val

In [11]:
x_train, x_val, y_train, y_val = load_and_clean()

In [12]:
dtt([x_train])

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
,int64,category,category,int64,int64,bool,category,int64,bool,float64,category
386,42,M,ASY,145,0,False,Normal,99,True,0.000,Flat
886,52,M,NAP,138,223,False,Normal,169,False,0.000,Up
593,64,M,ASY,130,258,True,LVH,130,False,0.000,Flat
164,52,F,ATA,140,225,False,Normal,140,False,0.000,Up
228,41,M,ATA,120,295,False,Normal,170,False,0.000,Up


In [13]:
float_cols = UKS_NP_PD.numeric_cols(x_train)
cat_cols = UKS_NP_PD.cat_cols(x_train)
power_transformer = PowerTransformer(method="yeo-johnson", standardize=True)
one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

column_transform = ColumnTransformer(
    transformers=[("scaler", power_transformer, float_cols), ("onehot", one_hot, cat_cols)],
    # transformers=[("scaler", power_transformer, float_cols), ("ordinal", ordinal, cat_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False,
    verbose=True,
)
steps = [("column_transform", column_transform)]
pipeline_x = PipelinePD(steps=steps)

In [14]:
pipeline_y = PipelinePD([("identity", FunctionTransformer(lambda x: x))])

In [15]:
loader_heart = HeartNNLoader(UKS_CONSTANTS.KAGGLEHUB_DIR, pipeline_pd_xy=(pipeline_x, pipeline_y))

Saving hparams, ignoring ('pipeline_pd_xy',)


In [16]:
loader_heart.hparams_initial

"data_dir": /Users/Akseldkw/coding/data_kretsinger/kagglehub
"split":    None

In [17]:
_ = loader_heart.setup("fit")

Setting up data for stage: fit
Removed 0 rows, representing 0.00% of the data
[ColumnTransformer] ........ (1 of 2) Processing scaler, total=   0.0s
[ColumnTransformer] ........ (2 of 2) Processing onehot, total=   0.0s
Setting up data for stage: validate
Removed 0 rows, representing 0.00% of the data


In [18]:
tensor = torch.tensor

In [19]:
loader_heart._train.tensors[1].shape

torch.Size([734])

In [20]:
loader_heart.set_dataloader_args(batch_size=256, shuffle=True)

In [21]:
out = loader_heart.train_dataloader()

In [22]:
dtt([*loader_heart.x_y_processed])

Age 
 RestingBP 
 Cholesterol 
 MaxHR 
 Oldpeak 
 Sex_F 
 Sex_M 
 ChestPainType_ASY 
 ChestPainType_ATA 
 ChestPainType_NAP 
 ChestPainType_TA 
 FastingBS_False 
 FastingBS_True 
 RestingECG_LVH 
 RestingECG_Normal 
 RestingECG_ST 
 ExerciseAngina_False 
 ExerciseAngina_True 
 ST_Slope_Down 
 ST_Slope_Flat 
 ST_Slope_Up 
 
 
 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 
 
 
 
 273 
 0.154 
 -0.670 
 0.402 
 0.026 
 -0.805 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 
 
 285 
 -0.268 
 -1.194 
 0.217 
 -0.537 
 -0.805 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 
 
 511 
 1.026 
 1.473 
 0.673 
 -1.805 
 1.142 
 0.000 
 1.000 
 1.000 
 0.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 
 
 770 
 -0.783 
 -1.454 
 0.305 
 1.569 
 -0.805 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 
 
 858 
 0.915 
 0.392 
 0.678 
 1.079 
 2.148 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 
 
 
 
 
 
 HeartDisease 
 
 
 
 bool 
 
 
 
 
 273 
 False 
 
 
 285 
 False 
 
 
 511 
 True 
 
 
 770 
 False 
 
 
 858 
 True

In [23]:
x_process = loader_heart.x_y_processed[0]
y_process = loader_heart.x_y_processed[1]
x_process.shape, y_process.shape

((918, 21), (918, 1))

# Implementation

In [24]:
dtt(x_train)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
,int64,category,category,int64,int64,bool,category,int64,bool,float64,category
243,43,F,ATA,120,266,False,Normal,118,False,0.000,Up
587,37,M,NAP,118,240,False,LVH,165,False,1.000,Flat
212,56,M,NAP,130,276,False,Normal,128,True,1.000,Up
607,53,M,ASY,144,300,True,ST,128,True,1.500,Flat
70,57,M,ATA,140,265,False,ST,145,True,1.000,Flat


In [25]:
from kret_lightning.trainer_defaults import Trainer___init___TypedDict

static_args: Trainer___init___TypedDict = {
    "max_epochs": 50,  # Enough to see trends, not full convergence
    "limit_train_batches": 0.5,  # Use 50% of training data per epoch
    # "limit_val_batches": 0.5,  # Use 50% of val data (need reliable signal)
    # "log_every_n_steps": 50,  # Reduce logging overhead
    "enable_model_summary": False,  # Skip summary printout each trial
    "enable_checkpointing": False,  # No checkpoints during sweep (saves I/O)
    "gradient_clip_val": 1.0,  # Stability for exploring LR ranges
    "max_time": {"minutes": 30},  # Kill runaway trials
}

In [26]:
nn = HeartFailureNN()

Saving hparams, ignoring ['input_size']


In [31]:
TrainerDynamicDefaults.trainer_dynamic_defaults(nn, loader_heart, logtype=None)

{'logger': None,
 'default_root_dir': PosixPath('/Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_004'),
 'callbacks': []}

In [32]:
direction = "minimize"
topn_saver = TopNModelSaver(n=3, save_dir=nn.ckpt_path, direction=direction)

In [33]:
study = optuna.create_study(
    study_name="heart_failure_4",
    direction=direction,
    **OptunaDefaults.CREATE_STUDY_DEFAULTS,
)

[I 2026-01-29 13:49:56,419] Using an existing study with name 'heart_failure_4' instead of creating a new one.


In [34]:
def objective(trial: optuna.Trial) -> float:
    # Suggest hyperparameters
    hidden_size1 = trial.suggest_int("hidden_size1", 16, 128, step=16)
    hidden_size2 = trial.suggest_int("hidden_size2", 16, 128, step=16)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    l1 = trial.suggest_float("l1", 1e-5, 1e-2, log=True)
    l2 = trial.suggest_float("l2", 1e-5, 1e-2, log=True)

    model = HeartFailureNN(
        hidden_sizes=[hidden_size1, hidden_size2],
        dropout_rate=dropout_rate,
        lr=lr,
        l1_penalty=l1,
        l2_penalty=l2,
    )
    dynamic_args = TrainerDynamicDefaults.trainer_dynamic_defaults(model, loader_heart, logtype=None, trial=trial)
    trainer_args = static_args | dynamic_args

    trainer = L.Trainer(**trainer_args)  # New trainer per trial!
    assert trainer.logger is not None
    # trainer.logger.log_hyperparams(model.hparams_initial)
    trainer.fit(model, datamodule=loader_heart, **TrainerStaticDefaults.TRAINER_FIT)

    score = trainer.callback_metrics["val_loss"].item()
    topn_saver.maybe_save(trainer, model, score, trial)
    return score

In [35]:
OptunaDefaults.OPTIM_STUDY_DEF

{'n_jobs': 1, 'gc_after_trial': True, 'show_progress_bar': True}

In [36]:
optim_args = OptunaDefaults.study_n_trials(10) | OptunaDefaults.OPTIM_STUDY_DEF
# optim_args["n_jobs"] = 1
optim_args

{'n_trials': 10,
 'timeout': None,
 'n_jobs': 1,
 'gc_after_trial': True,
 'show_progress_bar': True}

In [37]:
nn.ckpt_path, nn._ckpt_pattern_tuple.filename

(PosixPath('/Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_004'),
 'best-{epoch:02d}-{val_loss:.2f}')

In [38]:
study.optimize(objective, **optim_args)

  0%|          | 0/10 [00:00<?, ?it/s]

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
Loading `train_dataloader` to estimate number of stepping batches.


Saving hparams, ignoring ['input_size']
Setting up data for stage: TrainerFn.FITTING
Removed 0 rows, representing 0.00% of the data
Setting up data for stage: validate
Removed 0 rows, representing 0.00% of the data


Output()

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


...

Metric val_loss improved by 0.000 >= min_delta = 0.0001. New best score: 0.673


Metric val_loss improved by 0.000 >= min_delta = 0.0001. New best score: 0.673


Metric val_loss improved by 0.000 >= min_delta = 0.0001. New best score: 0.673


`Trainer.fit` stopped: `max_epochs=50` reached.


[I 2026-01-29 13:52:09,952] Trial 9 finished with value: 0.6726447939872742 and parameters: {'hidden_size1': 112, 'hidden_size2': 112, 'dropout_rate': 0.15203848783007287, 'lr': 0.00012769270936423784, 'l1': 2.181039835267928e-05, 'l2': 8.631980698797511e-05}. Best is trial 3 with value: 0.43340444564819336.


# Sandbox

In [39]:
topn_saver._leaderboard

[(0.4379233717918396,
  0,
  PosixPath('/Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_004/trial_0000_score_0.4379.ckpt')),
 (0.4443143904209137,
  1,
  PosixPath('/Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_004/trial_0001_score_0.4443.ckpt')),
 (0.43340444564819336,
  3,
  PosixPath('/Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_004/trial_0003_score_0.4334.ckpt'))]